# Day 12: Vision Transformer (ViT), Tokenization & Self-Attention 🛰️

## 1. Learning Objective & Educational Context

On **Day 8**, we implemented our baseline **Hyperspectral 3D-CNN** (`Hyperspectral3DCNN`), which achieved **97.02% Overall Accuracy** on Day 11's held-out Salinas test split. 

Today (**Day 12**), we transition from purely convolutional architectures to **attention-based modeling** by implementing and understanding a **Vision Transformer (ViT)** tailored for hyperspectral patches.

### Why Attention After the 3D-CNN?
1. **Local Receptive Fields vs. Global Pairwise Affinity**:
   - Convolutions compute features within fixed, localized kernels (e.g. $3 \times 3 \times 3$). While effective for local textures, convolutions require deep networks to propagate spatial context across the entire patch.
   - **Self-Attention** computes pairwise relationships directly between all spatial coordinates: token $i$ can attend to token $j$ anywhere in the patch in a single operation.
2. **Roadmap Positioning**:
   - **Day 12 (Today)**: Understand spatial tokenization, learnable positional embeddings, multi-head self-attention, and verify tensor compatibility via forward pass without training.
   - **Day 13 (Tomorrow)**: Fuse 3D-CNN local representations with Transformer global attention representations.
   - **Day 14**: Train the hybrid model on the real Salinas dataset.

> **Strict Scope Boundary**: Today is dedicated solely to architecture verification and self-attention interpretation. We do not train the model, tune hyperparameters, or alter Day 11 evaluation results.

## 2. Environment Setup & Hyperspectral Input Structure

We import our modules from `terraspectra.models.vit` and inspect the dimensions of real hyperspectral patches from the Salinas data pipeline.

In [1]:
import sys
from pathlib import Path

# Resolve project root dynamically whether run from root or notebooks/
current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn

from terraspectra.models.vit import (
    HyperspectralViT,
    HyperspectralViTConfig,
    PatchTokenizer,
    TransformerEncoderBlock,
    build_vit,
)
from terraspectra.data.dataset import create_dataloaders
from terraspectra.data.loader import SALINAS_CLASS_NAMES

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

PyTorch Version: 2.14.1+cpu
Device: CPU


### Inspecting Input Tensor Dimensions

The standard batch emitted by our Day 9 PyTorch DataLoaders has shape:
$$\text{Batch Shape} = (B, C, D, H, W)$$

- $B$: Batch size (e.g., 16 samples)
- $C$: Feature channels ($C=1$ for hyperspectral cubes)
- $D$: Spectral depth ($D=15$ PCA components or $D=204$ calibrated AVIRIS bands)
- $H, W$: Spatial window diameter ($7 \times 7$ patch centered on the pixel to classify)

Let's load a real sample batch from the PCA-reduced Salinas dataset.

In [2]:
# Load PCA-reduced Salinas cube and ground-truth labels
pca_path = PROJECT_ROOT / "data" / "processed" / "salinas_pca_reduced.npz"
gt_path = PROJECT_ROOT / "data" / "raw" / "Salinas_gt.mat"

if pca_path.is_file() and gt_path.is_file():
    pca_data = np.load(pca_path)
    cube_pca = pca_data["reduced_cube"] if "reduced_cube" in pca_data else pca_data["cube"]
    gt = sio.loadmat(gt_path)["salinas_gt"]  # (512, 217)
    
    # Create DataLoaders using established Day 9 pipeline
    train_loader, val_loader, test_loader, meta = create_dataloaders(
        cube_or_dataset=cube_pca,
        gt=gt,
        patch_size=7,
        batch_size=16,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=42,
    )
    
    # Fetch a representative test batch
    sample_batch_x, sample_batch_y = next(iter(test_loader))
    print(f"Loaded real Salinas batch:")
    print(f"  Input Features shape: {sample_batch_x.shape} (dtype: {sample_batch_x.dtype})")
    print(f"  Target Labels shape:   {sample_batch_y.shape} (dtype: {sample_batch_y.dtype})")
    print(f"  Target classes in batch: {torch.unique(sample_batch_y).tolist()}")
else:
    # Synthetic fallback fixture
    sample_batch_x = torch.randn(16, 1, 15, 7, 7)
    sample_batch_y = torch.randint(0, 16, (16,))
    print(f"Generated synthetic batch: {sample_batch_x.shape}")

Loaded real Salinas batch:
  Input Features shape: torch.Size([16, 1, 15, 7, 7]) (dtype: torch.float32)
  Target Labels shape:   torch.Size([16]) (dtype: torch.int64)
  Target classes in batch: [2, 3, 5, 6, 7, 8, 9, 11, 14]


## 3. Hyperspectral Tokenization: Patches to Tokens

In natural vision (2D RGB images), ViT (Dosovitskiy et al., 2020) cuts a large $224 \times 224$ image into non-overlapping $16 \times 16$ patches, treating each patch as a token.

In **hyperspectral remote sensing**, each sample is already a small local patch ($7 \times 7$) centered on a specific pixel. 

### Our Tokenization Strategy: Spatial Tokens
1. We treat each spatial coordinate $(h, w)$ within the $7 \times 7$ window as an **individual spatial token**.
2. For each spatial pixel, its feature vector consists of its **entire spectral signature** ($C \times D = 15$ values).
3. A learnable linear projection maps the 15-dimensional spectral feature vector into a latent **embedding dimension** $E = 64$.
4. Output sequence: $(B, N, E)$ where $N = H \times W = 7 \times 7 = 49$ tokens.

```text
Input Volume (B, 1, 15, 7, 7)
      │
      ▼ Permute & Reshape
Spatial Sequence (B, 49, 15)  [49 pixels, each with 15 spectral bands]
      │
      ▼ Linear Projection (15 -> 64)
Token Embeddings (B, 49, 64)  [49 latent tokens of dimension E=64]
```

### Design Trade-Off
- **Advantage**: Preserves explicit 2D spatial arrangement, allowing self-attention to learn how canopy pixels interact with neighboring soil or weed pixels.
- **Alternative (Spectral Tokens)**: Tokenizing every wavelength band would produce $204 \times 49 \approx 10,000$ tokens, leading to an intractable quadratic memory explosion ($O(N^2)$ attention matrix). Spatial tokenization with spectral projection is both memory-efficient and physically interpretable.

In [3]:
tokenizer = PatchTokenizer(in_channels=1, spectral_depth=15, embed_dim=64)
sample_tokens = tokenizer(sample_batch_x)

print(f"Input hyperspectral patch shape: {sample_batch_x.shape}")
print(f"Tokenized sequence shape:        {sample_tokens.shape}")
print(f"Number of spatial tokens (N):     {sample_tokens.shape[1]} (7x7 pixels)")
print(f"Token embedding dimension (E):    {sample_tokens.shape[2]}")

Input hyperspectral patch shape: torch.Size([16, 1, 15, 7, 7])
Tokenized sequence shape:        torch.Size([16, 49, 64])
Number of spatial tokens (N):     49 (7x7 pixels)
Token embedding dimension (E):    64


## 4. Positional Embeddings: Encoding Spatial Coordinates

Self-attention is **permutation-equivariant**:
$$\text{Attention}(P X, P X, P X) = P \,\text{Attention}(X, X, X)$$

If we randomly permuted the 49 spatial tokens, self-attention without positional information would produce the exact same attention affinity values. 

In remote sensing, **relative spatial position is vital**: the center pixel $(3, 3)$ is the target to be classified, while surrounding pixels provide spatial context (e.g., vineyard rows, crop borders).

To encode spatial position, we add **learnable 1D/2D positional embeddings**:
$$X_{\text{positioned}} = X_{\text{tokens}} + E_{\text{pos}}$$
where $E_{\text{pos}} \in \mathbb{R}^{1 \times N \times E}$.

In [4]:
# Instantiate positional embeddings
pos_embedding = nn.Parameter(torch.randn(1, 49, 64) * 0.02)
tokens_with_pos = sample_tokens + pos_embedding

print(f"Token embeddings:       {sample_tokens.shape}")
print(f"Positional embeddings:  {pos_embedding.shape}")
print(f"Tokens + Pos Embedding: {tokens_with_pos.shape}")

Token embeddings:       torch.Size([16, 49, 64])
Positional embeddings:  torch.Size([1, 49, 64])
Tokens + Pos Embedding: torch.Size([16, 49, 64])


## 5. Multi-Head Self-Attention: Mathematical Foundations

The core of the Transformer is **Scaled Dot-Product Self-Attention**:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

### The Roles of Q, K, V:
1. **Query ($Q = X W_Q$)**: What each token is *looking for* or querying from the patch.
2. **Key ($K = X W_K$)**: What each token *advertises* or provides as context.
3. **Value ($V = X W_V$)**: The substantive information/content to be aggregated.
4. **Dot Product ($Q K^T$)**: Measures raw compatibility/affinity between token $i$ and token $j$.
5. **Scaling ($\frac{1}{\sqrt{d_k}}$)**: For embedding dimension $E=64$ and $h=4$ heads, key dimension $d_k = 64/4 = 16$. Scaling by $\sqrt{16} = 4$ prevents large dot products from pushing the softmax into vanishing gradient regions.
6. **Softmax Normalization**: Converts affinity scores into a probability distribution where row $\sum_{j=1}^{49} A_{i, j} = 1.0$.
7. **Multi-Head Splitting**: Splits $E=64$ into $h=4$ separate sub-spaces of dimension $d_k=16$. Each head can independently attend to different spatial patterns (e.g., Head 0 attends to center, Head 1 attends to edges, Head 2 attends to row patterns).

In [5]:
# Step-by-step Scaled Dot-Product Attention verification
B, N, E = sample_tokens.shape
num_heads = 4
d_k = E // num_heads

# Linear projections for one sample
sample_x = tokens_with_pos[:1]  # (1, 49, 64)

W_q = nn.Linear(E, E)
W_k = nn.Linear(E, E)
W_v = nn.Linear(E, E)

Q = W_q(sample_x).view(1, N, num_heads, d_k).transpose(1, 2)  # (1, num_heads, N, d_k)
K = W_k(sample_x).view(1, N, num_heads, d_k).transpose(1, 2)
V = W_v(sample_x).view(1, N, num_heads, d_k).transpose(1, 2)

# Compute scaled dot-product attention scores
scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)  # (1, num_heads, N, N)
attn_weights_step = torch.softmax(scores, dim=-1)

print(f"Q, K, V shapes:            {Q.shape}")
print(f"Attention scores shape:    {scores.shape}")
print(f"Attention weights shape:   {attn_weights_step.shape}")
print(f"Row sums check (softmax):  {attn_weights_step[0, 0, 0].sum().item():.4f} (must equal 1.0)")

Q, K, V shapes:            torch.Size([1, 4, 49, 16])
Attention scores shape:    torch.Size([1, 4, 49, 49])
Attention weights shape:   torch.Size([1, 4, 49, 49])
Row sums check (softmax):  1.0000 (must equal 1.0)


### Visualizing Spatial Attention Across Attention Heads

Let's extract the actual self-attention weights from our `HyperspectralViT` model on our Salinas batch.

For a $7 \times 7$ patch, the **center pixel** (which corresponds to the ground-truth label) is at index **24** (row 3, col 3 in 0-indexed coordinates). 

By extracting row 24 from the $49 \times 49$ attention matrix, we obtain how strongly the center pixel attends to all 49 spatial locations across the patch!

In [6]:
# Instantiate HyperspectralViT and extract attention weights
config = HyperspectralViTConfig(
    in_channels=1,
    spectral_depth=15,
    spatial_size=7,
    num_classes=16,
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    mlp_dim=128,
    pooling="mean",
)
model = HyperspectralViT(config)
model.eval()

# Retrieve attention weights for the first sample in batch
with torch.inference_mode():
    logits, attn_layers = model(sample_batch_x[:1], return_attention=True)

# Layer 0 attention weights: shape (1, 4, 49, 49)
layer0_attn = attn_layers[0][0]  # (4 heads, 49, 49)

center_pixel_idx = 24  # (3, 3) in a 7x7 patch

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f"Spatial Attention of Center Pixel (Token 24) Across 4 Heads (Layer 0)", fontsize=14, fontweight="bold")

for head_idx in range(4):
    # Extract row 24: attention from center pixel to all 49 pixels
    center_attn = layer0_attn[head_idx, center_pixel_idx].view(7, 7).numpy()
    
    im = axes[head_idx].imshow(center_attn, cmap="viridis", interpolation="nearest")
    axes[head_idx].set_title(f"Head {head_idx}")
    axes[head_idx].set_xlabel("Col Index (0..6)")
    axes[head_idx].set_ylabel("Row Index (0..6)")
    # Mark the center pixel with a red cross
    axes[head_idx].plot(3, 3, "r+", markersize=14, markeredgewidth=2)
    fig.colorbar(im, ax=axes[head_idx], fraction=0.046, pad=0.04)

plt.tight_layout()
fig_dir = PROJECT_ROOT / "outputs" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "vit_spatial_attention_heads.png", dpi=150)
plt.show()

print(f"Saved attention map visualization to {fig_dir / 'vit_spatial_attention_heads.png'}")

Saved attention map visualization to C:\Users\abhis\Desktop\Tetraspectra\outputs\figures\vit_spatial_attention_heads.png


C:\Users\abhis\AppData\Local\Temp\ipykernel_7228\3767000268.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. The Transformer Encoder Block

We use a modern **Pre-LayerNorm** Transformer block:
$$X^{(1)} = X + \text{Dropout}(\text{MHA}(\text{LayerNorm}(X)))$$
$$X^{(2)} = X^{(1)} + \text{Dropout}(\text{MLP}(\text{LayerNorm}(X^{(1)})))$$

### Why Pre-LayerNorm?
- In Post-LayerNorm (original 2017 Transformer), gradients flow through unnormalized residual additions, requiring precise learning rate warmups.
- In Pre-LayerNorm, the residual stream remains identity-mapped, allowing smooth gradient flow directly from the loss back to the earliest layers.

### Multi-Layer Perceptron (MLP)
- Two linear projections: $E=64 \to \text{mlp\_dim}=128 \to E=64$.
- Activation: **GELU** (Gaussian Error Linear Unit), providing smooth non-linear feature transformation.
- Dropout: 0.1 for regularization.

In [7]:
block = TransformerEncoderBlock(embed_dim=64, num_heads=4, mlp_dim=128, dropout_rate=0.1)
out_block, _ = block(tokens_with_pos, need_weights=False)

print(f"Input to block:  {tokens_with_pos.shape}")
print(f"Output of block: {out_block.shape}")
assert out_block.shape == tokens_with_pos.shape, "Transformer block must preserve sequence and embedding dimensions"
print("[SUCCESS] Block dimension preservation verified.")

Input to block:  torch.Size([16, 49, 64])
Output of block: torch.Size([16, 49, 64])
[SUCCESS] Block dimension preservation verified.


## 7. Full ViT Forward Pass & Parameter Count

We now execute the full `HyperspectralViT` on the 16-sample Salinas batch.

### Classification Head & Pooling
- **Token Pooling**: We use **mean pooling** over all 49 spatial tokens: $\bar{z} = \frac{1}{N} \sum_{i=1}^N z_i$.
- **LayerNorm**: Normalizes pooled embedding: $\tilde{z} = \text{LayerNorm}(\bar{z})$.
- **Linear Classifier**: Maps $\tilde{z} \in \mathbb{R}^{64} \to \text{logits} \in \mathbb{R}^{16}$.
- **Unnormalized Logits**: The output contains raw logits (no softmax) for standard compatibility with `nn.CrossEntropyLoss`.

In [8]:
# Instantiate model via factory
vit_model = build_vit(
    spectral_depth=15,
    spatial_size=7,
    num_classes=16,
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    mlp_dim=128,
    pooling="mean",
)
vit_model.eval()

# Forward pass
with torch.inference_mode():
    logits = vit_model(sample_batch_x)

print(f"Input batch shape:    {sample_batch_x.shape}")
print(f"Output logits shape:  {logits.shape} (Expected: ({sample_batch_x.shape[0]}, 16))")
print(f"Finite values check:  {torch.isfinite(logits).all().item()}")

# Parameter breakdown
vit_params = vit_model.count_parameters()
print("\nParameter Counts:")
print(f"  Total Parameters:       {vit_params['total_parameters']:,}")
print(f"  Trainable Parameters:   {vit_params['trainable_parameters']:,}")
print(f"  Non-Trainable:          {vit_params['non_trainable_parameters']:,}")

Input batch shape:    torch.Size([16, 1, 15, 7, 7])
Output logits shape:  torch.Size([16, 16]) (Expected: (16, 16))
Finite values check:  True

Parameter Counts:
  Total Parameters:       72,272
  Trainable Parameters:   72,272
  Non-Trainable:          0


### Architectural Comparison: 3D-CNN vs. Vision Transformer

| Architectural Property | Baseline 3D-CNN (Day 8) | Hyperspectral ViT (Day 12) |
| :--- | :--- | :--- |
| **Primary Mechanism** | 3D Spatial-Spectral Convolutions | Scaled Dot-Product Multi-Head Self-Attention |
| **Receptive Field** | Local ($3 \times 3 \times 3$ kernel stages) | Global ($49 \times 49$ pairwise affinity across patch) |
| **Inductive Bias** | Strong translation equivariance & locality | Weak spatial inductive bias (relies on pos embeddings) |
| **Parameter Count** | ~23,352 parameters | ~72,272 parameters |
| **Spectral Handling** | 3D Conv kernels slide along spectral depth | Linear projection of spectral vector into latent $E=64$ |
| **Compute Complexity** | $O(B \cdot C \cdot D \cdot H \cdot W \cdot K^3)$ | $O(B \cdot N^2 \cdot E)$ where $N=H \times W$ |

## 8. Architectural Limitations & Scientific Caution

> [!WARNING]
> **Forward Pass $\neq$ Trained Accuracy**: A successful forward pass confirms that tensor layouts, matrix multiplications, sequence permutations, and classification heads are mathematically compatible and numerically stable. It does **NOT** prove that the model can accurately classify Salinas land cover. Untrained transformer weights produce random output logits.

### Fundamental Trade-Offs in Hyperspectral Transformers:
1. **Data Hunger**: Transformers lack the localized inductive bias inherent in convolutions. On small remote sensing datasets without massive pre-training, pure Transformers often overfit or perform slightly worse than CNNs.
2. **Spatial vs. Spectral Attention**: Our spatial tokenization treats the 15 spectral bands as a single projected feature vector. It explicitly models spatial relationships, but does not compute band-to-band spectral self-attention.
3. **Motivation for Day 13 (CNN-ViT Hybrid Fusion)**:
   - Convolutions extract rich, local, fine-grained edge and texture representations.
   - Transformers capture global context and long-range canopy correlations.
   - Combining both into a **Hybrid CNN-Transformer** (Day 13) leverages the strengths of both paradigms!

## 9. What I Learned & Interview Discussion Points

### Key Takeaways for ML Engineering:
1. **Hyperspectral Tokenization**: Converting a 5D tensor $(B, 1, D, H, W)$ into a token sequence $(B, H \cdot W, E)$ maps continuous spectral curves into latent embedding vectors at each spatial pixel.
2. **Positional Embeddings are Mandatory**: Self-attention is permutation-equivariant; without learnable positional embeddings, the model cannot distinguish between center target pixels and boundary neighbor pixels.
3. **Scaled Dot-Product Formula**: Dividing $Q K^T$ by $\sqrt{d_k}$ prevents the dot products from growing excessively large, ensuring softmax gradients remain informative and stable.
4. **Pre-LN vs. Post-LN**: Pre-LayerNorm enables direct identity gradient flow through residual connections, drastically stabilizing training.
5. **Next Step**: On **Day 13**, we will design the **CNN-ViT Hybrid Fusion Network**, combining our Day 8 3D-CNN feature extractor with this Day 12 Transformer encoder!